# Exercise 5.1: Approximate Inference

In this excerise we look at three methods of using approximate inference using pgmpy:
- Stochastic Sampling
- Likelihood weighted sampling
- Rejection sampling

## Stochastic Sampling

These examples use a simplified version of the student model described in Exercise 3.5 without the SAT or letter variables.
In the first case samples are taken from P(X | parents(X)) given the values of Parents(X). No evidence is used.

The number of samples produced should correspond to the CPDs of the network. 

In [1]:
#This code is modified from the examples at pgmpy.org Copyright (c) 2013-2021 pgmpy
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.sampling import BayesianModelSampling

student = BayesianNetwork([('diff', 'grade'), ('intel', 'grade')])
cpd_d = TabularCPD('diff', 2, [[0.6], [0.4]])
cpd_i = TabularCPD('intel', 2, [[0.7], [0.3]])
cpd_g = TabularCPD('grade', 3, [[0.3, 0.05, 0.9, 0.5], [0.4, 0.25,
               0.08, 0.3], [0.3, 0.7, 0.02, 0.2]],
               ['intel', 'diff'], [2, 2])
student.add_cpds(cpd_d, cpd_i, cpd_g)
inference = BayesianModelSampling(student)
sampled = inference.forward_sample(size=2)


  0%|          | 0/3 [00:00<?, ?it/s]

Adapt the number of samples taken and use the sampled dataframe returned to calculate the estimates for the CPDs. 

In [2]:
#For example by summing the diff column we can see how many samples were taken with diff=1
sampled['intel'].sum()

1

## Likelihood weighted

Generates weighted sample(s) from joint distribution of the bayesian network, that comply with the given evidence. ‘Probabilistic Graphical Model Principles and Techniques’, Koller and Friedman, Algorithm 12.2 pp 493.

In this example we specify the evidence the course was easy (diff = 0).

In [3]:
from pgmpy.factors.discrete import State
from pgmpy.models import BayesianModel
from pgmpy.factors.discrete import TabularCPD
from pgmpy.sampling import BayesianModelSampling
student = BayesianModel([('diff', 'grade'), ('intel', 'grade')])
cpd_d = TabularCPD('diff', 2, [[0.6], [0.4]])
cpd_i = TabularCPD('intel', 2, [[0.7], [0.3]])
cpd_g = TabularCPD('grade', 3, [[0.3, 0.05, 0.9, 0.5], [0.4, 0.25,
        0.08, 0.3], [0.3, 0.7, 0.02, 0.2]],
        ['intel', 'diff'], [2, 2])
student.add_cpds(cpd_d, cpd_i, cpd_g)
inference = BayesianModelSampling(student)
evidence = [State('diff', 0)]
inference.likelihood_weighted_sample(evidence=evidence, size=2)


C:\Users\wesle\AppData\Roaming\Python\Python39\site-packages\pgmpy\models\BayesianModel.py:8: FutureWarning: BayesianModel has been renamed to BayesianNetwork. Please use BayesianNetwork class, BayesianModel will be removed in future.
  warnings.warn(


  0%|          | 0/3 [00:00<?, ?it/s]

,diff,grade,intel,_weight
0,0,1,0,0.6
1,0,2,0,0.6


As before we can use the samples returned to estimate the posterori probability P(X|evidence). Try changing the evidence you use to see the effect.
As the evidence variables increase the performance will degrade.

## Rejection Sampling

In this case P(X|e) is estimated only from samples that agree with e

In [4]:
from pgmpy.models import BayesianModel
from pgmpy.factors.discrete import TabularCPD
from pgmpy.factors.discrete import State
from pgmpy.sampling import BayesianModelSampling
student = BayesianModel([('diff', 'grade'), ('intel', 'grade')])
cpd_d = TabularCPD('diff', 2, [[0.6], [0.4]])
cpd_i = TabularCPD('intel', 2, [[0.7], [0.3]])
cpd_g = TabularCPD('grade', 3, [[0.3, 0.05, 0.9, 0.5], [0.4, 0.25,
               0.08, 0.3], [0.3, 0.7, 0.02, 0.2]],
               ['intel', 'diff'], [2, 2])
student.add_cpds(cpd_d, cpd_i, cpd_g)
inference = BayesianModelSampling(student)
evidence = [State(var='diff', state=0)]
inference.rejection_sample(evidence=evidence, size=2)

C:\Users\wesle\AppData\Roaming\Python\Python39\site-packages\pgmpy\models\BayesianModel.py:8: FutureWarning: BayesianModel has been renamed to BayesianNetwork. Please use BayesianNetwork class, BayesianModel will be removed in future.
  warnings.warn(


  0%|          | 0/2 [00:00<?, ?it/s]

,diff,grade,intel
0,0,1,0
1,0,1,0


This becomes hopelessly expensive if P(e) is small. See if you can demonstrate that by changing the CPDs of the network.